# 2. Clean

Even though dbt already has its own tests on this mart (`dbt/gridcast/models/marts/_marts.yml`), this pipeline doesn't trust the export blindly -- it re-checks for the two problems that actually matter downstream:

1. **Duplicate rows** -- re-running the ingest export can land the same `(respondent, hour)` twice.
2. **Negative demand readings** -- physically impossible; a handful show up as sensor/reporting glitches in the EIA feed.

Missing values (e.g. `demand_lag_168h` for a region's first week of history) are left as NaN on purpose -- that's expected warm-up behavior, not something to fill in here. Rows still missing what they need get dropped later, in the featurize stage.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd

from src.ml.clean import clean_demand_forecast

raw_path = PROJECT_ROOT / "data" / "raw" / "features_raw.csv"
df = pd.read_csv(raw_path, parse_dates=["demand_hour_utc"])
df.shape

(131402, 16)

### Before cleaning

In [3]:
n_duplicates = df.duplicated(subset=["respondent", "demand_hour_utc"]).sum()
n_negative = (df["demand_mwh"] < 0).sum()
print(f"duplicate (respondent, hour) rows: {n_duplicates}")
print(f"negative demand_mwh readings:       {n_negative}")

duplicate (respondent, hour) rows: 0
negative demand_mwh readings:       0


In [4]:
df_clean = clean_demand_forecast(df)

### After cleaning

In [5]:
print(f"rows before: {len(df):,}")
print(f"rows after:  {len(df_clean):,}")

rows before: 131,402
rows after:  131,402


In [6]:
out_path = PROJECT_ROOT / "data" / "interim" / "features_clean.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(out_path, index=False)
print(f"wrote {out_path}")

wrote /Users/devashish/Desktop/Projects/gridcast/data/interim/features_clean.csv
